# 01 · Data and preprocessing

Builds the filtered, normalised spatial / scRNA-seq pair for one dataset and saves it to `data/processed_gene_data/`.

1. Load the MERFISH section and the scRNA-seq reference (`DATA.md` lists the files)
2. Keep the cortex only: subcortical cells are dropped because the reference has low gene throughput there
3. Lower-case gene names, `normalize_total` + `log1p`, keep shared genes, annotate per-gene sparsity

Run once per dataset (`DATASET = "d1"` / `"d2"`). Needs the `impute` extras (TISSUE).

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st

from spatial_imputation import config, data, metrics, stats, plotting
plotting.set_style()

In [ ]:
DATASET = "d1"   # "d1" or "d2"
spec = config.get_dataset(DATASET)
print(spec.description)

## Load raw data

In [ ]:
spatial_raw, reference_raw = data.load_raw(DATASET)
spatial_raw, reference_raw

## Cortex filter
Ellipse parameters live in `config.DATASETS[...].cortex` and were tuned by eye for each section.

In [ ]:
spatial = data.filter_cortex(spatial_raw, DATASET)
print(f"kept {spatial.n_obs:,} / {spatial_raw.n_obs:,} cells")
plotting.plot_cells(spatial);

## Sequencing depth
Transcripts per cell in the MERFISH section vs UMIs per cell in the reference (before normalisation).

In [ ]:
display(data.dataset_summary(spatial, reference_raw).round(1))
plotting.plot_count_depth(spatial, reference_raw);

## Normalise and match genes
> **Note (d2):** `SHG101350_processed.h5ad` is already log-normalised, so this step normalises it a second time. The reported results used this; see the README.

In [ ]:
spatial, reference = data.preprocess_pair(spatial, reference_raw, normalize=True)
print(f"{spatial.n_vars} shared genes")
del spatial_raw, reference_raw

In [ ]:
gene = "nos1"
fig, ax = plt.subplots(figsize=(8, 6))
plotting.spatial_scatter(ax, spatial, plotting.display_values(spatial[:, gene].X), title=gene);

## Store Tangram training genes and save

In [ ]:
from spatial_imputation.imputation import prepare_tangram

prepare_tangram(spatial, reference)   # adds uns['training_genes']
data.save_processed(DATASET, spatial, reference)
config.processed_paths(DATASET)